SCD

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
SCD_TABLE = "fpl_project.silver.player_price_history"

incoming = spark.table("fpl_project.silver.players") \
    .select("player_id", "display_name", "current_price", "team_id")

if not spark.catalog.tableExists(SCD_TABLE):
    # First run: initialize with all current prices
    scd_init = incoming \
        .withColumn("effective_from", F.current_date()) \
        .withColumn("effective_to", F.lit("9999-12-31").cast("date")) \
        .withColumn("is_current", F.lit(True))

    scd_init.write.format("delta").saveAsTable(SCD_TABLE)
    print("SCD table initialized.")
else:
    # Find players whose price has changed
    current_records = spark.table(SCD_TABLE).filter("is_current = true")

    changes = incoming.alias("s").join(
        current_records.alias("t"),
        "player_id"
    ).filter("s.current_price != t.current_price")

    change_count = changes.count()

    if change_count > 0:
        target = DeltaTable.forName(spark, SCD_TABLE)

        # Step 1: Close out old records (set end date and mark not current)
        target.alias("t").merge(
            changes.select(F.col("s.player_id").alias("player_id")).alias("c"),
            "t.player_id = c.player_id AND t.is_current = true"
        ).whenMatchedUpdate(set={
            "effective_to": "current_date()",
            "is_current": "false"
        }).execute()

        # Step 2: Insert new current records with today's date
        new_records = changes.select(
            F.col("s.player_id"),
            F.col("s.display_name"),
            F.col("s.current_price"),
            F.col("s.team_id"),
            F.current_date().alias("effective_from"),
            F.lit("9999-12-31").cast("date").alias("effective_to"),
            F.lit(True).alias("is_current"),
        )

        new_records.write.format("delta").mode("append") \
            .saveAsTable(SCD_TABLE)

        print(f"SCD updated: {change_count} price changes recorded.")
    else:
        print("No price changes detected.")